In [1]:
import random
import numpy as np
import torch
import torchvision

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

print("\nPyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Device:", device)
print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Image shape:", train_dataset[0][0].shape)
print("Classes:", train_dataset.classes)

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 173kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.18MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 3.85MB/s]


PyTorch version: 2.11.0+cu130
Torchvision version: 0.26.0+cu130
Device: cuda
Training samples: 60000
Test samples: 10000
Image shape: torch.Size([1, 28, 28])
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


### Train/Validation Split

Split the original Fashion-MNIST training set into training and validation subsets using a fixed random seed.

Keep the official test set untouched for final evaluation and use DataLoaders for efficient batch processing.

In [2]:
from torch.utils.data import random_split, DataLoader

train_subset, val_subset = random_split(
    train_dataset,
    lengths=[50000, 10000],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_subset,
    batch_size=128,
    shuffle=True,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_subset,
    batch_size=128,
    shuffle=False,
    pin_memory=(device.type == "cuda")
)

print("Training samples:", len(train_subset))
print("Validation samples:", len(val_subset))
print("Test samples:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training samples: 50000
Validation samples: 10000
Test samples: 10000
Training batches: 391
Validation batches: 79


### Deep MLP Architecture

Define an eight-hidden-layer Multilayer Perceptron (MLP) for Fashion-MNIST classification.

Each hidden layer contains 256 neurons with ReLU activation. The model is designed to expose intermediate representations for future early-exit classifiers.

In [3]:
import torch.nn as nn

class DeepMLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256,
                 num_layers=8, num_classes=10):
        super().__init__()

        self.hidden_layers = nn.ModuleList()

        for i in range(num_layers):
            in_features = input_dim if i == 0 else hidden_dim

            self.hidden_layers.append(
                nn.Sequential(
                    nn.Linear(in_features, hidden_dim),
                    nn.ReLU()
                )
            )

        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, return_features=False):
        x = x.flatten(start_dim=1)
        features = {}

        for i, layer in enumerate(self.hidden_layers, start=1):
            x = layer(x)

            if i in (2, 4, 6):
                features[i] = x

        logits = self.classifier(x)

        if return_features:
            return logits, features

        return logits


model = DeepMLP().to(device)

dummy_input = torch.randn(4, 1, 28, 28).to(device)

with torch.no_grad():
    logits, features = model(dummy_input, return_features=True)

print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Final output shape:", logits.shape)

for layer_idx, feature in features.items():
    print(f"Layer {layer_idx} features:", feature.shape)

Model parameters: 664,074
Final output shape: torch.Size([4, 10])
Layer 2 features: torch.Size([4, 256])
Layer 4 features: torch.Size([4, 256])
Layer 6 features: torch.Size([4, 256])


### Baseline Model Training

Train the eight-layer MLP using the Adam optimizer and cross-entropy loss.

Monitor training loss and validation accuracy across epochs. Save the model weights corresponding to the best validation accuracy for subsequent early-exit experiments.

In [4]:
import copy
import time
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20
patience = 5

best_val_accuracy = 0.0
best_model_state = None
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(num_epochs):
    start_time = time.time()

    # Training
    model.train()
    running_train_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)

    train_loss = running_train_loss / len(train_subset)

    # Validation
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    val_loss = running_val_loss / total
    val_accuracy = correct / total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4%} | "
        f"Time: {elapsed:.1f}s"
    )

    # Early stopping based on validation accuracy
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break

# Restore the best model
model.load_state_dict(best_model_state)

# Save baseline weights
torch.save(
    model.state_dict(),
    "mlp_baseline_best.pth"
)

print()
print(f"Best Validation Accuracy: {best_val_accuracy:.4%}")
print("Saved model: mlp_baseline_best.pth")

Epoch 01/20 | Train Loss: 0.8845 | Val Loss: 0.6583 | Val Accuracy: 74.6800% | Time: 8.6s
Epoch 02/20 | Train Loss: 0.5119 | Val Loss: 0.4497 | Val Accuracy: 83.3100% | Time: 7.4s
Epoch 03/20 | Train Loss: 0.4172 | Val Loss: 0.3863 | Val Accuracy: 86.6000% | Time: 8.1s
Epoch 04/20 | Train Loss: 0.3727 | Val Loss: 0.3747 | Val Accuracy: 86.9800% | Time: 7.5s
Epoch 05/20 | Train Loss: 0.3403 | Val Loss: 0.3639 | Val Accuracy: 86.8700% | Time: 7.7s
Epoch 06/20 | Train Loss: 0.3294 | Val Loss: 0.3923 | Val Accuracy: 86.9400% | Time: 7.9s
Epoch 07/20 | Train Loss: 0.3064 | Val Loss: 0.3565 | Val Accuracy: 87.4800% | Time: 7.2s
Epoch 08/20 | Train Loss: 0.2939 | Val Loss: 0.3335 | Val Accuracy: 88.0300% | Time: 7.9s
Epoch 09/20 | Train Loss: 0.2836 | Val Loss: 0.3562 | Val Accuracy: 87.1000% | Time: 7.7s
Epoch 10/20 | Train Loss: 0.2796 | Val Loss: 0.3520 | Val Accuracy: 87.8000% | Time: 7.6s
Epoch 11/20 | Train Loss: 0.2712 | Val Loss: 0.3392 | Val Accuracy: 87.9600% | Time: 8.0s
Epoch 12/2

### Early-Exit Classifiers

Attach auxiliary classification heads to hidden layers 2, 4, and 6 of the pretrained MLP.

Freeze the backbone parameters and train only the early-exit heads. This preserves the original baseline and allows us to evaluate the predictive quality of intermediate representations.

In [5]:
class EarlyExitMLP(nn.Module):
    def __init__(self, backbone, hidden_dim=256, num_classes=10):
        super().__init__()

        self.backbone = backbone

        # Freeze the pretrained backbone
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

        # Auxiliary classification heads
        self.exit_heads = nn.ModuleDict({
            "2": nn.Linear(hidden_dim, num_classes),
            "4": nn.Linear(hidden_dim, num_classes),
            "6": nn.Linear(hidden_dim, num_classes)
        })

    def forward(self, x):
        final_logits, features = self.backbone(
            x, return_features=True
        )

        exit_logits = {
            layer: self.exit_heads[str(layer)](features[layer])
            for layer in (2, 4, 6)
        }

        exit_logits[8] = final_logits

        return exit_logits


early_exit_model = EarlyExitMLP(model).to(device)

# Keep the frozen backbone in evaluation mode
early_exit_model.backbone.eval()

dummy_input = torch.randn(4, 1, 28, 28).to(device)

with torch.no_grad():
    outputs = early_exit_model(dummy_input)

print("Early-exit output shapes:")

for layer, logits in outputs.items():
    print(f"Layer {layer}: {logits.shape}")

trainable_parameters = sum(
    p.numel()
    for p in early_exit_model.parameters()
    if p.requires_grad
)

print()
print("Trainable parameters:", f"{trainable_parameters:,}")

Early-exit output shapes:
Layer 2: torch.Size([4, 10])
Layer 4: torch.Size([4, 10])
Layer 6: torch.Size([4, 10])
Layer 8: torch.Size([4, 10])

Trainable parameters: 7,710


### Training Early-Exit Classifiers

Train the auxiliary classifiers at hidden layers 2, 4, and 6 while keeping the pretrained backbone frozen.

Use cross-entropy loss for each exit and monitor validation accuracy independently. Restore the best-performing heads based on their mean validation accuracy.

In [6]:
import copy
import torch.optim as optim

exit_layers = (2, 4, 6)

optimizer_exits = optim.Adam(
    early_exit_model.exit_heads.parameters(),
    lr=0.001
)

num_epochs_exits = 15

best_mean_accuracy = 0.0
best_exit_state = None

exit_history = {
    layer: [] for layer in exit_layers
}

for epoch in range(num_epochs_exits):

    # Train auxiliary heads only
    early_exit_model.train()
    early_exit_model.backbone.eval()

    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_exits.zero_grad()

        outputs = early_exit_model(images)

        losses = [
            criterion(outputs[layer], labels)
            for layer in exit_layers
        ]

        loss = torch.stack(losses).mean()

        loss.backward()
        optimizer_exits.step()

        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_subset)

    # Validate each early-exit head
    early_exit_model.eval()

    correct = {layer: 0 for layer in exit_layers}
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = early_exit_model(images)

            for layer in exit_layers:
                predictions = outputs[layer].argmax(dim=1)
                correct[layer] += (
                    predictions == labels
                ).sum().item()

            total += labels.size(0)

    accuracies = {
        layer: correct[layer] / total
        for layer in exit_layers
    }

    mean_accuracy = sum(accuracies.values()) / len(exit_layers)

    for layer in exit_layers:
        exit_history[layer].append(accuracies[layer])

    if mean_accuracy > best_mean_accuracy:
        best_mean_accuracy = mean_accuracy
        best_exit_state = copy.deepcopy(
            early_exit_model.exit_heads.state_dict()
        )

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs_exits} | "
        f"Loss: {train_loss:.4f} | "
        f"Exit 2: {accuracies[2]:.2%} | "
        f"Exit 4: {accuracies[4]:.2%} | "
        f"Exit 6: {accuracies[6]:.2%}"
    )

# Restore the best heads
early_exit_model.exit_heads.load_state_dict(best_exit_state)
early_exit_model.eval()

torch.save(
    early_exit_model.state_dict(),
    "mlp_early_exit_best.pth"
)

print()
print("Best mean validation accuracy:", f"{best_mean_accuracy:.2%}")
print("Saved model: mlp_early_exit_best.pth")

Epoch 01/15 | Loss: 0.5021 | Exit 2: 88.92% | Exit 4: 89.38% | Exit 6: 89.23%
Epoch 02/15 | Loss: 0.2387 | Exit 2: 89.16% | Exit 4: 89.29% | Exit 6: 89.30%
Epoch 03/15 | Loss: 0.2095 | Exit 2: 89.26% | Exit 4: 89.30% | Exit 6: 89.43%
Epoch 04/15 | Loss: 0.1972 | Exit 2: 89.28% | Exit 4: 89.21% | Exit 6: 89.40%
Epoch 05/15 | Loss: 0.1905 | Exit 2: 89.39% | Exit 4: 89.31% | Exit 6: 89.35%
Epoch 06/15 | Loss: 0.1856 | Exit 2: 89.51% | Exit 4: 89.30% | Exit 6: 89.39%
Epoch 07/15 | Loss: 0.1823 | Exit 2: 89.45% | Exit 4: 89.27% | Exit 6: 89.29%
Epoch 08/15 | Loss: 0.1798 | Exit 2: 89.59% | Exit 4: 89.34% | Exit 6: 89.40%
Epoch 09/15 | Loss: 0.1778 | Exit 2: 89.70% | Exit 4: 89.35% | Exit 6: 89.44%
Epoch 10/15 | Loss: 0.1764 | Exit 2: 89.74% | Exit 4: 89.39% | Exit 6: 89.45%
Epoch 11/15 | Loss: 0.1747 | Exit 2: 89.74% | Exit 4: 89.40% | Exit 6: 89.45%
Epoch 12/15 | Loss: 0.1737 | Exit 2: 89.72% | Exit 4: 89.38% | Exit 6: 89.36%
Epoch 13/15 | Loss: 0.1728 | Exit 2: 89.74% | Exit 4: 89.39% | E

### Early-Exit Validation Analysis

Evaluate the restored early-exit model at layers 2, 4, 6, and 8 using the validation dataset.

Compare prediction accuracy and analyze disagreements between the earliest exit and the full network to identify opportunities and risks for adaptive computation.

In [7]:
import pandas as pd

early_exit_model.eval()

exit_layers_all = (2, 4, 6, 8)

predictions = {layer: [] for layer in exit_layers_all}
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device, non_blocking=True)

        outputs = early_exit_model(images)

        for layer in exit_layers_all:
            predictions[layer].append(
                outputs[layer].argmax(dim=1).cpu()
            )

        all_labels.append(labels)

y_true = torch.cat(all_labels)

predictions = {
    layer: torch.cat(predictions[layer])
    for layer in exit_layers_all
}

results = []

for layer in exit_layers_all:
    accuracy = (
        predictions[layer] == y_true
    ).float().mean().item()

    results.append({
        "Exit Layer": layer,
        "Accuracy": round(accuracy * 100, 3)
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

correct_2 = predictions[2] == y_true
correct_8 = predictions[8] == y_true

print()
print("Exit 2 wrong, Exit 8 correct:",
      ((~correct_2) & correct_8).sum().item())

print("Exit 2 correct, Exit 8 wrong:",
      (correct_2 & (~correct_8)).sum().item())

print("Both wrong:",
      ((~correct_2) & (~correct_8)).sum().item())

print("Both correct:",
      (correct_2 & correct_8).sum().item())

 Exit Layer  Accuracy
          2     89.76
          4     89.42
          6     89.47
          8     89.13

Exit 2 wrong, Exit 8 correct: 136
Exit 2 correct, Exit 8 wrong: 199
Both wrong: 888
Both correct: 8777


### Confidence-Based Early-Exit Routing

Evaluate a confidence-based routing policy across the intermediate classifiers.

For each input, select the earliest exit whose maximum softmax probability exceeds a specified threshold. Compare classification accuracy, exit distribution, and estimated computational savings against the full-depth baseline.

In [8]:
import pandas as pd
import torch

early_exit_model.eval()

exit_layers = (2, 4, 6, 8)
thresholds = [0.50, 0.70, 0.80, 0.90, 0.95, 0.99]

all_logits = {layer: [] for layer in exit_layers}
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device, non_blocking=True)

        outputs = early_exit_model(images)

        for layer in exit_layers:
            all_logits[layer].append(outputs[layer].cpu())

        all_labels.append(labels)

all_logits = {
    layer: torch.cat(all_logits[layer])
    for layer in exit_layers
}

y_true = torch.cat(all_labels)
num_samples = len(y_true)

results = []

for threshold in thresholds:
    selected_predictions = torch.empty(
        num_samples, dtype=torch.long
    )

    selected_layers = torch.full(
        (num_samples,), 8, dtype=torch.long
    )

    undecided = torch.ones(
        num_samples, dtype=torch.bool
    )

    for layer in (2, 4, 6):
        probabilities = torch.softmax(
            all_logits[layer], dim=1
        )

        confidence, predictions = probabilities.max(dim=1)

        exit_mask = undecided & (confidence >= threshold)

        selected_predictions[exit_mask] = predictions[exit_mask]
        selected_layers[exit_mask] = layer

        undecided[exit_mask] = False

    selected_predictions[undecided] = (
        all_logits[8][undecided].argmax(dim=1)
    )

    accuracy = (
        selected_predictions == y_true
    ).float().mean().item()

    average_depth = selected_layers.float().mean().item()

    # Depth-based proxy, not measured FLOPs or latency
    estimated_saving = 1 - average_depth / 8

    results.append({
        "Threshold": threshold,
        "Accuracy (%)": round(accuracy * 100, 3),
        "Avg Exit Layer": round(average_depth, 3),
        "Depth Saving (%)": round(estimated_saving * 100, 2),
        "Exit 2 (%)": round(
            (selected_layers == 2).float().mean().item() * 100, 2
        ),
        "Exit 4 (%)": round(
            (selected_layers == 4).float().mean().item() * 100, 2
        ),
        "Exit 6 (%)": round(
            (selected_layers == 6).float().mean().item() * 100, 2
        ),
        "Exit 8 (%)": round(
            (selected_layers == 8).float().mean().item() * 100, 2
        )
    })

routing_results_df = pd.DataFrame(results)

print(routing_results_df.to_string(index=False))

 Threshold  Accuracy (%)  Avg Exit Layer  Depth Saving (%)  Exit 2 (%)  Exit 4 (%)  Exit 6 (%)  Exit 8 (%)
      0.50         89.69           2.110             73.63       96.83        1.80        0.41        0.96
      0.70         89.48           2.544             68.20       88.57        3.11        0.87        7.45
      0.80         89.42           2.825             64.69       83.72        3.48        0.65       12.15
      0.90         89.27           3.187             60.16       77.44        3.75        0.84       17.97
      0.95         89.21           3.474             56.58       72.54        4.03        0.64       22.79
      0.99         89.17           4.043             49.47       62.33        5.07        0.74       31.86


### Confidence and Prediction Error Analysis

Analyze the confidence distributions of correct and incorrect predictions at the earliest exit.

Evaluate whether maximum softmax probability can distinguish prediction errors and identify overconfident mistakes that may undermine confidence-based routing.

In [9]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

logits_2 = all_logits[2]

probabilities = torch.softmax(logits_2, dim=1)
confidence, predictions_2 = probabilities.max(dim=1)

correct_mask = predictions_2 == y_true
error_mask = ~correct_mask

correct_confidence = confidence[correct_mask].numpy()
error_confidence = confidence[error_mask].numpy()

error_detection_auc = roc_auc_score(
    error_mask.numpy().astype(int),
    1 - confidence.numpy()
)

summary = pd.DataFrame({
    "Metric": [
        "Correct predictions",
        "Incorrect predictions",
        "Mean confidence (correct)",
        "Mean confidence (incorrect)",
        "Median confidence (correct)",
        "Median confidence (incorrect)",
        "Incorrect predictions with confidence >= 0.90",
        "Incorrect predictions with confidence >= 0.99",
        "Error detection AUROC"
    ],
    "Value": [
        int(correct_mask.sum()),
        int(error_mask.sum()),
        round(correct_confidence.mean(), 4),
        round(error_confidence.mean(), 4),
        round(np.median(correct_confidence), 4),
        round(np.median(error_confidence), 4),
        int((confidence[error_mask] >= 0.90).sum()),
        int((confidence[error_mask] >= 0.99).sum()),
        round(error_detection_auc, 4)
    ]
})

print(summary.to_string(index=False))

                                       Metric     Value
                          Correct predictions 8976.0000
                        Incorrect predictions 1024.0000
                    Mean confidence (correct)    0.9467
                  Mean confidence (incorrect)    0.6915
                  Median confidence (correct)    0.9996
                Median confidence (incorrect)    0.6844
Incorrect predictions with confidence >= 0.90  206.0000
Incorrect predictions with confidence >= 0.99   55.0000
                        Error detection AUROC    0.9021


### Confidence and the Value of Continued Computation

Analyze whether the confidence at the earliest exit can distinguish samples that benefit from deeper computation from those that are harmed by it.

Compare confidence distributions for corrected errors and newly introduced errors, and quantify how frequently deeper processing changes the prediction.

In [10]:
import numpy as np
import pandas as pd

# Predictions from the previously evaluated exits
pred_2 = predictions[2]
pred_8 = predictions[8]

# Confidence at the earliest exit
prob_2 = torch.softmax(all_logits[2], dim=1)
confidence_2 = prob_2.max(dim=1).values

# Define outcome groups
corrected = (pred_2 != y_true) & (pred_8 == y_true)
harmed = (pred_2 == y_true) & (pred_8 != y_true)
both_correct = (pred_2 == y_true) & (pred_8 == y_true)
both_wrong = (pred_2 != y_true) & (pred_8 != y_true)

groups = {
    "Corrected by deeper layers": corrected,
    "Harmed by deeper layers": harmed,
    "Both correct": both_correct,
    "Both wrong": both_wrong
}

results = []

for name, mask in groups.items():
    values = confidence_2[mask].numpy()

    results.append({
        "Group": name,
        "Samples": len(values),
        "Mean Confidence": round(float(np.mean(values)), 4),
        "Median Confidence": round(float(np.median(values)), 4),
        "Confidence >= 0.90 (%)": round(
            float(np.mean(values >= 0.90) * 100), 2
        )
    })

analysis_df = pd.DataFrame(results)

print(analysis_df.to_string(index=False))

print()
print(
    "Predictions changed:",
    int((pred_2 != pred_8).sum())
)

print(
    "Net accuracy gain from always continuing:",
    round(
        ((pred_8 == y_true).float().mean()
         - (pred_2 == y_true).float().mean()).item() * 100,
        3
    ),
    "percentage points"
)

                     Group  Samples  Mean Confidence  Median Confidence  Confidence >= 0.90 (%)
Corrected by deeper layers     1027           0.9919             1.0000                   97.57
   Harmed by deeper layers      979           0.9792             1.0000                   94.18
              Both correct        0              NaN                NaN                     NaN
                Both wrong     7994           0.9042             0.9947                   72.80

Predictions changed: 1
Net accuracy gain from always continuing: 0.48 percentage points


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


### Validate Early-Exit Prediction Consistency

Reconstruct predictions directly from the cached validation logits to avoid variable overwriting or inconsistent results.

Verify the confusion groups and compare the earliest exit against the final classifier before proceeding with further experiments.

In [11]:
import torch

# Reconstruct predictions from the same cached logits
pred_2 = all_logits[2].argmax(dim=1)
pred_8 = all_logits[8].argmax(dim=1)

labels = y_true

assert len(pred_2) == len(pred_8) == len(labels) == 10000

# Correctness masks
correct_2 = pred_2 == labels
correct_8 = pred_8 == labels

# Mutually exclusive outcome groups
corrected = (~correct_2) & correct_8
harmed = correct_2 & (~correct_8)
both_correct = correct_2 & correct_8
both_wrong = (~correct_2) & (~correct_8)

print("Exit 2 accuracy:", f"{correct_2.float().mean():.2%}")
print("Exit 8 accuracy:", f"{correct_8.float().mean():.2%}")

print()
print("Corrected by deeper layers:", corrected.sum().item())
print("Harmed by deeper layers:", harmed.sum().item())
print("Both correct:", both_correct.sum().item())
print("Both wrong:", both_wrong.sum().item())

print()
print("Predictions changed:", (pred_2 != pred_8).sum().item())

group_total = (
    corrected.sum()
    + harmed.sum()
    + both_correct.sum()
    + both_wrong.sum()
).item()

print("Group total:", group_total)

assert group_total == len(labels), "Inconsistent outcome groups!"

Exit 2 accuracy: 89.76%
Exit 8 accuracy: 89.13%

Corrected by deeper layers: 136
Harmed by deeper layers: 199
Both correct: 8777
Both wrong: 888

Predictions changed: 393
Group total: 10000


### Confidence and the Value of Deeper Computation

Compare early-exit confidence across samples corrected or harmed by deeper processing.

Determine whether confidence alone can identify inputs that benefit from additional computation, providing a baseline for future reliability-aware routing.

In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Recompute confidence from cached logits
confidence_2 = torch.softmax(
    all_logits[2], dim=1
).max(dim=1).values

groups = {
    "Corrected by deeper layers": corrected,
    "Harmed by deeper layers": harmed,
    "Both correct": both_correct,
    "Both wrong": both_wrong
}

results = []

for name, mask in groups.items():
    values = confidence_2[mask].numpy()

    results.append({
        "Group": name,
        "Samples": len(values),
        "Mean Confidence": round(float(values.mean()), 4),
        "Median Confidence": round(float(np.median(values)), 4),
        "Confidence >= 0.90 (%)": round(
            float((values >= 0.90).mean() * 100), 2
        )
    })

confidence_analysis_df = pd.DataFrame(results)

print(confidence_analysis_df.to_string(index=False))

# Evaluate confidence specifically on samples where
# deeper computation changes correctness
changed_correctness = corrected | harmed

benefit_labels = corrected[changed_correctness].numpy().astype(int)
benefit_scores = 1 - confidence_2[changed_correctness].numpy()

benefit_auc = roc_auc_score(
    benefit_labels,
    benefit_scores
)

print()
print("Benefit Detection AUROC:", round(benefit_auc, 4))
print("Prediction disagreements:", int((pred_2 != pred_8).sum()))

                     Group  Samples  Mean Confidence  Median Confidence  Confidence >= 0.90 (%)
Corrected by deeper layers      136           0.5543             0.5434                    0.74
   Harmed by deeper layers      199           0.5956             0.5649                    5.03
              Both correct     8777           0.9547             0.9997                   85.77
                Both wrong      888           0.7125             0.7231                   23.09

Benefit Detection AUROC: 0.5665
Prediction disagreements: 393
